# Multi-Style Image Generator with Ice Crystal Effects

This notebook creates a Gradio app for Hugging Face Spaces that generates styled images using textual inversion embeddings with optional ice crystal overlay effects.

**Compatible with Google Colab** - Uses Colab secrets for HF authentication.

## Features
- **5 Predefined Styles**: 8bit, ahx_beta, dr_strange, max_naylor, smiling_friend
- **Custom Style Upload**: Upload your own `.bin` embedding files
- **Ice Crystal Effect**: Optional crystalline overlay with adjustable intensity
- **Configurable Parameters**: Seed and guidance scale controls

## Colab Setup
1. Open this notebook in Google Colab
2. Go to **Runtime > Change runtime type** and select **T4 GPU** (for faster local testing)
3. Click the **key icon** in the left sidebar to add secrets
4. Add a secret named `HF_TOKEN` with your Hugging Face write token

**Note:** The HF Space will be deployed on CPU (free tier). Your username is auto-detected from your token.

## 1. Install Dependencies

In [ ]:
%pip install -q torch diffusers transformers accelerate gradio huggingface_hub Pillow numpy tqdm scipy

## 2. Hugging Face Hub Login

Login to Hugging Face to push your app to Spaces.

**For Google Colab:** Add your HF token as a secret named `HF_TOKEN` in Colab (click the key icon in the left sidebar).

**For other environments:** You'll be prompted to enter your token interactively.

In [ ]:
from huggingface_hub import login, HfApi
import os

# Check if running in Google Colab
try:
    from google.colab import userdata
    # Use Colab secrets - add HF_TOKEN in Colab's secret manager (key icon in sidebar)
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print("Logged in using Colab secrets!")
except ImportError:
    # Not in Colab - use interactive login
    from huggingface_hub import notebook_login
    notebook_login()
except Exception as e:
    print(f"Error accessing Colab secrets: {e}")
    print("Please add HF_TOKEN to Colab secrets (key icon in left sidebar)")
    print("Or set it manually: login(token='your_token_here')")

## 2b. (Colab Only) Setup Style Files

Upload your style embedding files to Colab or mount Google Drive to access them.

In [ ]:
# Option 1: Mount Google Drive (if your style files are there)
# from google.colab import drive
# drive.mount('/content/drive')
# Then copy files: !cp /content/drive/MyDrive/your_styles/*.bin styles/

# Option 2: Create styles directory and upload files manually
import os
os.makedirs("styles", exist_ok=True)

# Check if running in Colab
try:
    from google.colab import files
    print("Running in Google Colab!")
    print("\nTo upload style files:")
    print("1. Run the cell below to upload .bin files")
    print("2. Or mount Google Drive and copy files to 'styles/' directory")
    print("\nStyles directory created at: /content/styles/")
except ImportError:
    print("Not running in Colab - ensure 'styles/' directory exists locally")

In [ ]:
# Upload style files (Colab only) - uncomment and run to upload .bin files
# from google.colab import files
# import shutil
# 
# uploaded = files.upload()
# for filename in uploaded.keys():
#     if filename.endswith('.bin'):
#         shutil.move(filename, f"styles/{filename}")
#         print(f"Moved {filename} to styles/")

# Verify styles directory contents
import os
if os.path.exists("styles"):
    style_files = [f for f in os.listdir("styles") if f.endswith('.bin')]
    if style_files:
        print("Style files found:")
        for f in style_files:
            print(f"  - styles/{f}")
    else:
        print("No .bin files found in styles/ directory")
        print("Please upload your style embedding files")
else:
    print("styles/ directory not found")

## 3. Define Ice Crystal Loss Function

This function calculates a loss to encourage transparent ice crystal patterns as an overlay effect.

In [ ]:
import torch
import torch.nn.functional as F

def ice_crystal_loss(images):
    """
    Calculate loss to encourage TRANSPARENT ice crystal patterns as an overlay.
    This version preserves the original content while adding crystalline effects.

    Args:
        images: Tensor of shape (batch, 3, height, width) in range [0, 1]

    Returns:
        Scalar loss value (lower = more ice crystal-like)
    """
    # 1. Edge Detection - Sharp crystalline structures (KEEP THIS STRONG)
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                           dtype=images.dtype, device=images.device).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
                           dtype=images.dtype, device=images.device).view(1, 1, 3, 3)

    edges_x = F.conv2d(images, sobel_x.repeat(3, 1, 1, 1), padding=1, groups=3)
    edges_y = F.conv2d(images, sobel_y.repeat(3, 1, 1, 1), padding=1, groups=3)
    edge_magnitude = torch.sqrt(edges_x**2 + edges_y**2)

    # We want sharp edges but only in certain areas (not everywhere)
    edge_threshold = 0.1
    strong_edges = torch.relu(edge_magnitude - edge_threshold)
    edge_loss = -strong_edges.mean()

    # 2. Selective Brightness - Only encourage brightness in edge regions
    edge_mask = (edge_magnitude > edge_threshold).float()
    brightness = images.mean(dim=1, keepdim=True)

    # Only brighten areas with edges (crystal formations)
    selective_brightness = brightness * edge_mask
    brightness_loss = -selective_brightness.mean() * 0.3

    # 3. High-frequency details - Crystalline patterns
    laplacian_kernel = torch.tensor([[0, -1, 0], [-1, 4, -1], [0, -1, 0]],
                                    dtype=images.dtype, device=images.device).view(1, 1, 3, 3)
    high_freq = F.conv2d(images, laplacian_kernel.repeat(3, 1, 1, 1), padding=1, groups=3)

    # Encourage high-frequency content but not too aggressively
    high_freq_loss = -torch.abs(high_freq).mean() * 0.5

    # 4. Subtle cool tones (don't force everything to be blue)
    r, g, b = images[:, 0], images[:, 1], images[:, 2]

    # Only encourage cool tones in bright areas (ice crystals)
    bright_mask = (brightness.squeeze(1) > 0.5).float()
    cool_tone_loss = (r * bright_mask).mean() - ((b * bright_mask).mean() + (g * bright_mask).mean()) / 2
    cool_tone_loss = cool_tone_loss * 0.2

    # 5. Texture variance - Ice crystals have varied texture
    kernel_size = 3
    local_mean = F.avg_pool2d(images, kernel_size, stride=1, padding=kernel_size//2)
    local_variance = F.avg_pool2d((images - local_mean)**2, kernel_size, stride=1, padding=kernel_size//2)

    # Encourage variance in edge regions (crystalline texture)
    texture_in_edges = local_variance * edge_mask.unsqueeze(1)
    texture_loss = -texture_in_edges.mean() * 0.5

    # Combine with BALANCED weights (preserve original content)
    total_loss = (
        3.0 * edge_loss +           # Sharp edges (most important)
        0.5 * brightness_loss +      # Selective brightness only
        0.8 * high_freq_loss +       # Crystalline details
        0.2 * cool_tone_loss +       # Subtle cool tones
        1.0 * texture_loss           # Crystalline texture
    )

    return total_loss

print("Ice crystal loss function defined!")

## 4. Define Generation Function

This function generates images using a style embedding with optional ice crystal guidance.

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

from diffusers import AutoencoderKL, UNet2DConditionModel, LMSDiscreteScheduler
from transformers import CLIPTextModel, CLIPTokenizer

# Global variables for models (will be loaded once)
vae = None
tokenizer = None
text_encoder = None
unet = None
scheduler = None
device = None

def load_models():
    """Load all models once and cache them globally."""
    global vae, tokenizer, text_encoder, unet, scheduler, device
    
    # Check if already loaded
    if vae is not None and scheduler is not None:
        return
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    model_id = "CompVis/stable-diffusion-v1-4"
    
    try:
        print("Loading models... (this may take a few minutes on CPU)")
        
        # Load with float16 on GPU, float32 on CPU
        dtype = torch.float16 if device == "cuda" else torch.float32
        
        vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", torch_dtype=dtype).to(device)
        tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
        text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=dtype).to(device)
        unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet", torch_dtype=dtype).to(device)
        
        # Initialize scheduler
        scheduler = LMSDiscreteScheduler(
            beta_start=0.00085,
            beta_end=0.012,
            beta_schedule="scaled_linear",
            num_train_timesteps=1000
        )
        
        print("Models loaded successfully!")
        
    except Exception as e:
        print(f"Error loading models: {e}")
        raise RuntimeError(f"Failed to load models: {e}")

def generate_with_style(
    style_file,
    prompt,
    seed=42,
    num_inference_steps=50,
    guidance_scale=7.5,
    height=512,
    width=512,
    use_ice_crystal_guidance=False,
    ice_crystal_loss_scale=50,
    guidance_frequency=10,
    progress=None
):
    """
    Generate an image using a style embedding with optional ice crystal guidance.

    Args:
        style_file: Path to the .bin file containing the learned embedding
        prompt: Text prompt (should include <style> placeholder)
        seed: Random seed for reproducibility
        num_inference_steps: Number of denoising steps
        guidance_scale: Classifier-free guidance scale
        height: Image height
        width: Image width
        use_ice_crystal_guidance: Whether to apply ice crystal pattern guidance
        ice_crystal_loss_scale: Scale for ice crystal loss
        guidance_frequency: Apply guidance every N steps
        progress: Gradio progress callback

    Returns:
        PIL Image
    """
    global vae, tokenizer, text_encoder, unet, scheduler, device
    
    # Ensure models are loaded
    load_models()
    
    # Set random seed
    generator = torch.Generator(device=device).manual_seed(seed)

    # Load the learned embedding
    learned_embeds_dict = torch.load(style_file, map_location=device, weights_only=True)

    # Extract the token string and embedding vector
    style_token = list(learned_embeds_dict.keys())[0]
    style_embedding = learned_embeds_dict[style_token].to(device)

    # Get expected embedding dimension from text encoder
    expected_dim = text_encoder.get_input_embeddings().weight.shape[1]

    # Handle dimension mismatch
    if style_embedding.shape[0] != expected_dim:
        if style_embedding.shape[0] == 1024 and expected_dim == 768:
            style_embedding = style_embedding[:768]
        else:
            raise ValueError(f"Cannot handle embedding dimension {style_embedding.shape[0]} -> {expected_dim}")

    # Add the token to the tokenizer if not already present
    if style_token not in tokenizer.get_vocab():
        tokenizer.add_tokens([style_token])
        text_encoder.resize_token_embeddings(len(tokenizer))

    # Get the token ID and inject embedding
    token_id = tokenizer.convert_tokens_to_ids(style_token)
    with torch.no_grad():
        text_encoder.get_input_embeddings().weight[token_id] = style_embedding

    # Replace the prompt placeholder with the actual style token
    final_prompt = prompt.replace("<style>", style_token)

    # Tokenize the prompt
    text_input = tokenizer(
        final_prompt,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt"
    )

    # Get text embeddings
    with torch.no_grad():
        text_embeddings = text_encoder(text_input.input_ids.to(device))[0]

    # Create unconditional embeddings for classifier-free guidance
    uncond_input = tokenizer(
        [""],
        padding="max_length",
        max_length=tokenizer.model_max_length,
        return_tensors="pt"
    )

    with torch.no_grad():
        uncond_embeddings = text_encoder(uncond_input.input_ids.to(device))[0]

    # Concatenate for classifier-free guidance
    text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

    # Initialize latents
    latents = torch.randn(
        (1, unet.config.in_channels, height // 8, width // 8),
        generator=generator,
        device=device
    )

    # Set scheduler timesteps
    scheduler.set_timesteps(num_inference_steps)
    latents = latents * scheduler.init_noise_sigma

    # Denoising loop
    for i, t in enumerate(tqdm(scheduler.timesteps, desc="Generating")):
        if progress:
            progress((i + 1) / num_inference_steps, f"Step {i + 1}/{num_inference_steps}")
            
        # Expand latents for classifier-free guidance
        latent_model_input = torch.cat([latents] * 2)
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        # Predict noise
        with torch.no_grad():
            noise_pred = unet(
                latent_model_input,
                t,
                encoder_hidden_states=text_embeddings
            ).sample

        # Perform classifier-free guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # Optional Ice Crystal Guidance
        if use_ice_crystal_guidance and i % guidance_frequency == 0:
            if device == "cuda":
                torch.cuda.empty_cache()

            latents = latents.detach().requires_grad_()
            sigma = scheduler.sigmas[i]
            latents_x0 = latents - sigma * noise_pred

            with torch.cuda.amp.autocast(enabled=False):
                denoised_images = vae.decode((1 / 0.18215) * latents_x0).sample / 2 + 0.5

            loss = ice_crystal_loss(denoised_images) * ice_crystal_loss_scale
            cond_grad = torch.autograd.grad(loss, latents)[0]
            latents = latents.detach() - cond_grad * sigma**2

            del denoised_images, loss, cond_grad
            if device == "cuda":
                torch.cuda.empty_cache()

        # Compute previous noisy sample
        latents = scheduler.step(noise_pred, t, latents).prev_sample

    # Decode latents to image
    latents = 1 / 0.18215 * latents

    with torch.no_grad():
        image = vae.decode(latents).sample

    # Convert to PIL Image
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.cpu().permute(0, 2, 3, 1).numpy()
    image = (image[0] * 255).astype(np.uint8)
    image = Image.fromarray(image)

    return image

print("Generation function defined!")

## 5. Build Gradio Interface

Create the Gradio app with style selection, custom upload, and ice crystal effect toggle.

In [ ]:
import gradio as gr

# Predefined styles mapping
PREDEFINED_STYLES = {
    "8bit": "styles/8bit_learned_embeds.bin",
    "ahx_beta": "styles/ahx_beta_learned_embeds.bin",
    "dr_strange": "styles/dr_strangelearned_embeds.bin",
    "max_naylor": "styles/max_naylorlearned_embeds.bin",
    "smiling_friend": "styles/smiling-friend-style_learned_embeds.bin"
}

def generate_image(
    prompt,
    style_choice,
    custom_embedding,
    seed,
    guidance_scale,
    use_ice_crystal,
    ice_crystal_intensity,
    progress=gr.Progress()
):
    """Main generation function for Gradio interface."""
    
    # Determine which style file to use
    if custom_embedding is not None:
        style_file = custom_embedding.name
    else:
        if style_choice not in PREDEFINED_STYLES:
            raise gr.Error("Please select a style or upload a custom embedding file.")
        style_file = PREDEFINED_STYLES[style_choice]
    
    # Check if file exists
    if not Path(style_file).exists():
        raise gr.Error(f"Style embedding file not found: {style_file}")
    
    try:
        image = generate_with_style(
            style_file=style_file,
            prompt=prompt,
            seed=int(seed),
            guidance_scale=guidance_scale,
            use_ice_crystal_guidance=use_ice_crystal,
            ice_crystal_loss_scale=ice_crystal_intensity,
            progress=progress
        )
        return image
    except Exception as e:
        raise gr.Error(f"Generation failed: {str(e)}")

# Build the Gradio interface
with gr.Blocks(
    title="Multi-Style Image Generator",
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="cyan"
    )
) as demo:
    gr.Markdown("""
    # Multi-Style Image Generator with Ice Crystal Effects
    
    Generate images using textual inversion style embeddings with optional ice crystal overlay effects.
    
    **Instructions:**
    1. Enter a prompt using `<style>` as placeholder (e.g., "A cat in the style of <style>")
    2. Select a predefined style OR upload your own `.bin` embedding file
    3. Optionally enable ice crystal effect for a crystalline overlay
    4. Click Generate!
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            # Input controls
            prompt = gr.Textbox(
                label="Prompt",
                placeholder="A mouse in the style of <style>",
                value="A mouse in the style of <style>",
                lines=2
            )
            
            style_choice = gr.Dropdown(
                choices=list(PREDEFINED_STYLES.keys()),
                value="8bit",
                label="Predefined Style",
                info="Select a bundled style embedding"
            )
            
            custom_embedding = gr.File(
                label="Custom Embedding (Optional)",
                file_types=[".bin"],
                type="filepath"
            )
            
            with gr.Row():
                seed = gr.Number(
                    label="Seed",
                    value=42,
                    precision=0
                )
                guidance_scale = gr.Slider(
                    label="Guidance Scale",
                    minimum=1.0,
                    maximum=20.0,
                    value=7.5,
                    step=0.5
                )
            
            with gr.Accordion("Ice Crystal Effect", open=False):
                use_ice_crystal = gr.Checkbox(
                    label="Enable Ice Crystal Effect",
                    value=False,
                    info="Add crystalline overlay to the image"
                )
                ice_crystal_intensity = gr.Slider(
                    label="Ice Crystal Intensity",
                    minimum=30,
                    maximum=100,
                    value=50,
                    step=5,
                    info="Higher = stronger crystal effect"
                )
            
            generate_btn = gr.Button("Generate", variant="primary", size="lg")
        
        with gr.Column(scale=1):
            # Output
            output_image = gr.Image(
                label="Generated Image",
                type="pil"
            )
    
    # Examples
    gr.Examples(
        examples=[
            ["A cat in the style of <style>", "8bit", None, 42, 7.5, False, 50],
            ["A mystical forest in the style of <style>", "dr_strange", None, 123, 7.5, False, 50],
            ["A portrait in the style of <style>", "max_naylor", None, 456, 7.5, True, 60],
        ],
        inputs=[prompt, style_choice, custom_embedding, seed, guidance_scale, use_ice_crystal, ice_crystal_intensity],
    )
    
    # Connect the generate button
    generate_btn.click(
        fn=generate_image,
        inputs=[prompt, style_choice, custom_embedding, seed, guidance_scale, use_ice_crystal, ice_crystal_intensity],
        outputs=output_image
    )

print("Gradio interface created!")

## 6. Write app.py

Write the complete app code to a file for Hugging Face Spaces deployment.

In [ ]:
%%writefile app.py
"""
Multi-Style Image Generator with Ice Crystal Effects
Hugging Face Spaces App
"""

import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import gradio as gr

from diffusers import AutoencoderKL, UNet2DConditionModel, LMSDiscreteScheduler
from transformers import CLIPTextModel, CLIPTokenizer

# Global variables for models (will be loaded once)
vae = None
tokenizer = None
text_encoder = None
unet = None
scheduler = None
device = None

# Predefined styles mapping
PREDEFINED_STYLES = {
    "8bit": "styles/8bit_learned_embeds.bin",
    "ahx_beta": "styles/ahx_beta_learned_embeds.bin",
    "dr_strange": "styles/dr_strangelearned_embeds.bin",
    "max_naylor": "styles/max_naylorlearned_embeds.bin",
    "smiling_friend": "styles/smiling-friend-style_learned_embeds.bin"
}


def ice_crystal_loss(images):
    """
    Calculate loss to encourage TRANSPARENT ice crystal patterns as an overlay.
    """
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                           dtype=images.dtype, device=images.device).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
                           dtype=images.dtype, device=images.device).view(1, 1, 3, 3)

    edges_x = F.conv2d(images, sobel_x.repeat(3, 1, 1, 1), padding=1, groups=3)
    edges_y = F.conv2d(images, sobel_y.repeat(3, 1, 1, 1), padding=1, groups=3)
    edge_magnitude = torch.sqrt(edges_x**2 + edges_y**2)

    edge_threshold = 0.1
    strong_edges = torch.relu(edge_magnitude - edge_threshold)
    edge_loss = -strong_edges.mean()

    edge_mask = (edge_magnitude > edge_threshold).float()
    brightness = images.mean(dim=1, keepdim=True)
    selective_brightness = brightness * edge_mask
    brightness_loss = -selective_brightness.mean() * 0.3

    laplacian_kernel = torch.tensor([[0, -1, 0], [-1, 4, -1], [0, -1, 0]],
                                    dtype=images.dtype, device=images.device).view(1, 1, 3, 3)
    high_freq = F.conv2d(images, laplacian_kernel.repeat(3, 1, 1, 1), padding=1, groups=3)
    high_freq_loss = -torch.abs(high_freq).mean() * 0.5

    r, g, b = images[:, 0], images[:, 1], images[:, 2]
    bright_mask = (brightness.squeeze(1) > 0.5).float()
    cool_tone_loss = (r * bright_mask).mean() - ((b * bright_mask).mean() + (g * bright_mask).mean()) / 2
    cool_tone_loss = cool_tone_loss * 0.2

    kernel_size = 3
    local_mean = F.avg_pool2d(images, kernel_size, stride=1, padding=kernel_size//2)
    local_variance = F.avg_pool2d((images - local_mean)**2, kernel_size, stride=1, padding=kernel_size//2)
    texture_in_edges = local_variance * edge_mask.unsqueeze(1)
    texture_loss = -texture_in_edges.mean() * 0.5

    total_loss = (
        3.0 * edge_loss +
        0.5 * brightness_loss +
        0.8 * high_freq_loss +
        0.2 * cool_tone_loss +
        1.0 * texture_loss
    )

    return total_loss


def load_models():
    """Load all models once and cache them globally."""
    global vae, tokenizer, text_encoder, unet, scheduler, device
    
    # Check if already loaded
    if vae is not None and scheduler is not None:
        return
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    model_id = "CompVis/stable-diffusion-v1-4"
    
    try:
        print("Loading models... (this may take a few minutes on CPU)")
        
        # Load with float16 on GPU, float32 on CPU
        dtype = torch.float16 if device == "cuda" else torch.float32
        
        vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", torch_dtype=dtype).to(device)
        tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
        text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=dtype).to(device)
        unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet", torch_dtype=dtype).to(device)
        
        # Initialize scheduler
        scheduler = LMSDiscreteScheduler(
            beta_start=0.00085,
            beta_end=0.012,
            beta_schedule="scaled_linear",
            num_train_timesteps=1000
        )
        
        print("Models loaded successfully!")
        
    except Exception as e:
        print(f"Error loading models: {e}")
        raise RuntimeError(f"Failed to load models: {e}")


def generate_with_style(
    style_file,
    prompt,
    seed=42,
    num_inference_steps=50,
    guidance_scale=7.5,
    height=512,
    width=512,
    use_ice_crystal_guidance=False,
    ice_crystal_loss_scale=50,
    guidance_frequency=10,
    progress=None
):
    """Generate an image using a style embedding with optional ice crystal guidance."""
    global vae, tokenizer, text_encoder, unet, scheduler, device
    
    load_models()
    
    generator = torch.Generator(device=device).manual_seed(seed)
    learned_embeds_dict = torch.load(style_file, map_location=device, weights_only=True)

    style_token = list(learned_embeds_dict.keys())[0]
    style_embedding = learned_embeds_dict[style_token].to(device)

    expected_dim = text_encoder.get_input_embeddings().weight.shape[1]

    if style_embedding.shape[0] != expected_dim:
        if style_embedding.shape[0] == 1024 and expected_dim == 768:
            style_embedding = style_embedding[:768]
        else:
            raise ValueError(f"Cannot handle embedding dimension {style_embedding.shape[0]} -> {expected_dim}")

    if style_token not in tokenizer.get_vocab():
        tokenizer.add_tokens([style_token])
        text_encoder.resize_token_embeddings(len(tokenizer))

    token_id = tokenizer.convert_tokens_to_ids(style_token)
    with torch.no_grad():
        text_encoder.get_input_embeddings().weight[token_id] = style_embedding

    final_prompt = prompt.replace("<style>", style_token)

    text_input = tokenizer(
        final_prompt,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        text_embeddings = text_encoder(text_input.input_ids.to(device))[0]

    uncond_input = tokenizer(
        [""],
        padding="max_length",
        max_length=tokenizer.model_max_length,
        return_tensors="pt"
    )

    with torch.no_grad():
        uncond_embeddings = text_encoder(uncond_input.input_ids.to(device))[0]

    text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

    latents = torch.randn(
        (1, unet.config.in_channels, height // 8, width // 8),
        generator=generator,
        device=device
    )

    scheduler.set_timesteps(num_inference_steps)
    latents = latents * scheduler.init_noise_sigma

    for i, t in enumerate(tqdm(scheduler.timesteps, desc="Generating")):
        if progress:
            progress((i + 1) / num_inference_steps, f"Step {i + 1}/{num_inference_steps}")
            
        latent_model_input = torch.cat([latents] * 2)
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        with torch.no_grad():
            noise_pred = unet(
                latent_model_input,
                t,
                encoder_hidden_states=text_embeddings
            ).sample

        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        if use_ice_crystal_guidance and i % guidance_frequency == 0:
            if device == "cuda":
                torch.cuda.empty_cache()

            latents = latents.detach().requires_grad_()
            sigma = scheduler.sigmas[i]
            latents_x0 = latents - sigma * noise_pred

            with torch.cuda.amp.autocast(enabled=False):
                denoised_images = vae.decode((1 / 0.18215) * latents_x0).sample / 2 + 0.5

            loss = ice_crystal_loss(denoised_images) * ice_crystal_loss_scale
            cond_grad = torch.autograd.grad(loss, latents)[0]
            latents = latents.detach() - cond_grad * sigma**2

            del denoised_images, loss, cond_grad
            if device == "cuda":
                torch.cuda.empty_cache()

        latents = scheduler.step(noise_pred, t, latents).prev_sample

    latents = 1 / 0.18215 * latents

    with torch.no_grad():
        image = vae.decode(latents).sample

    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.cpu().permute(0, 2, 3, 1).numpy()
    image = (image[0] * 255).astype(np.uint8)
    image = Image.fromarray(image)

    return image


def generate_image(
    prompt,
    style_choice,
    custom_embedding,
    seed,
    guidance_scale,
    use_ice_crystal,
    ice_crystal_intensity,
    progress=gr.Progress()
):
    """Main generation function for Gradio interface."""
    
    if custom_embedding is not None:
        style_file = custom_embedding
    else:
        if style_choice not in PREDEFINED_STYLES:
            raise gr.Error("Please select a style or upload a custom embedding file.")
        style_file = PREDEFINED_STYLES[style_choice]
    
    if not Path(style_file).exists():
        raise gr.Error(f"Style embedding file not found: {style_file}")
    
    try:
        image = generate_with_style(
            style_file=style_file,
            prompt=prompt,
            seed=int(seed),
            guidance_scale=guidance_scale,
            use_ice_crystal_guidance=use_ice_crystal,
            ice_crystal_loss_scale=ice_crystal_intensity,
            progress=progress
        )
        return image
    except Exception as e:
        raise gr.Error(f"Generation failed: {str(e)}")


# Build the Gradio interface
with gr.Blocks(
    title="Multi-Style Image Generator",
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="cyan"
    )
) as demo:
    gr.Markdown("""
    # Multi-Style Image Generator with Ice Crystal Effects
    
    Generate images using textual inversion style embeddings with optional ice crystal overlay effects.
    
    **Instructions:**
    1. Enter a prompt using `<style>` as placeholder (e.g., "A cat in the style of <style>")
    2. Select a predefined style OR upload your own `.bin` embedding file
    3. Optionally enable ice crystal effect for a crystalline overlay
    4. Click Generate!
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(
                label="Prompt",
                placeholder="A mouse in the style of <style>",
                value="A mouse in the style of <style>",
                lines=2
            )
            
            style_choice = gr.Dropdown(
                choices=list(PREDEFINED_STYLES.keys()),
                value="8bit",
                label="Predefined Style",
                info="Select a bundled style embedding"
            )
            
            custom_embedding = gr.File(
                label="Custom Embedding (Optional)",
                file_types=[".bin"],
                type="filepath"
            )
            
            with gr.Row():
                seed = gr.Number(
                    label="Seed",
                    value=42,
                    precision=0
                )
                guidance_scale = gr.Slider(
                    label="Guidance Scale",
                    minimum=1.0,
                    maximum=20.0,
                    value=7.5,
                    step=0.5
                )
            
            with gr.Accordion("Ice Crystal Effect", open=False):
                use_ice_crystal = gr.Checkbox(
                    label="Enable Ice Crystal Effect",
                    value=False,
                    info="Add crystalline overlay to the image"
                )
                ice_crystal_intensity = gr.Slider(
                    label="Ice Crystal Intensity",
                    minimum=30,
                    maximum=100,
                    value=50,
                    step=5,
                    info="Higher = stronger crystal effect"
                )
            
            generate_btn = gr.Button("Generate", variant="primary", size="lg")
        
        with gr.Column(scale=1):
            output_image = gr.Image(
                label="Generated Image",
                type="pil"
            )
    
    gr.Examples(
        examples=[
            ["A cat in the style of <style>", "8bit", None, 42, 7.5, False, 50],
            ["A mystical forest in the style of <style>", "dr_strange", None, 123, 7.5, False, 50],
            ["A portrait in the style of <style>", "max_naylor", None, 456, 7.5, True, 60],
        ],
        inputs=[prompt, style_choice, custom_embedding, seed, guidance_scale, use_ice_crystal, ice_crystal_intensity],
    )
    
    generate_btn.click(
        fn=generate_image,
        inputs=[prompt, style_choice, custom_embedding, seed, guidance_scale, use_ice_crystal, ice_crystal_intensity],
        outputs=output_image
    )

if __name__ == "__main__":
    demo.launch()

## 7. Write requirements.txt

Create the dependencies file for Hugging Face Spaces.

In [ ]:
%%writefile requirements.txt
torch
diffusers
transformers
accelerate
gradio
Pillow
numpy
tqdm
scipy

## 8. Push to Hugging Face Space

Create the Space and upload all necessary files. Your username will be auto-detected from your HF token.

1. Have your style embedding `.bin` files ready in a `styles/` directory
2. Run the cells below to create and populate the Space

In [ ]:
# Configuration
SPACE_NAME = "multi-style-generator"  # Name for your Space (you can change this)

# Auto-fetch username from HF token
from huggingface_hub import HfApi
api = HfApi()
user_info = api.whoami()
HF_USERNAME = user_info["name"]

# Full repo ID
REPO_ID = f"{HF_USERNAME}/{SPACE_NAME}"

print(f"Logged in as: {HF_USERNAME}")
print(f"Will create Space at: https://huggingface.co/spaces/{REPO_ID}")

In [ ]:
from huggingface_hub import create_repo

# Create the Space repository
# Using CPU for free tier (change to "t4-small" for GPU if you have a paid plan)
try:
    create_repo(
        repo_id=REPO_ID,
        repo_type="space",
        space_sdk="gradio",
        space_hardware="cpu-basic",  # Free tier CPU. Options: "cpu-basic", "cpu-upgrade", "t4-small", "t4-medium"
        private=False,
        exist_ok=True
    )
    print(f"Space created successfully: https://huggingface.co/spaces/{REPO_ID}")
    print("Note: Using CPU (free tier) - generation will be slower than GPU")
except Exception as e:
    print(f"Error creating Space: {e}")

In [ ]:
# Upload app.py and requirements.txt
from huggingface_hub import upload_file

# Upload app.py
api.upload_file(
    path_or_fileobj="app.py",
    path_in_repo="app.py",
    repo_id=REPO_ID,
    repo_type="space"
)
print("Uploaded: app.py")

# Upload requirements.txt
api.upload_file(
    path_or_fileobj="requirements.txt",
    path_in_repo="requirements.txt",
    repo_id=REPO_ID,
    repo_type="space"
)
print("Uploaded: requirements.txt")

In [ ]:
# Upload style embedding files
# Make sure your .bin files are in a 'styles/' directory locally

import os
from pathlib import Path

# Style files to upload (adjust paths as needed)
STYLE_FILES = {
    "styles/8bit_learned_embeds.bin": "8bit_learned_embeds.bin",
    "styles/ahx_beta_learned_embeds.bin": "ahx_beta_learned_embeds.bin",
    "styles/dr_strangelearned_embeds.bin": "dr_strangelearned_embeds.bin",
    "styles/max_naylorlearned_embeds.bin": "max_naylorlearned_embeds.bin",
    "styles/smiling-friend-style_learned_embeds.bin": "smiling-friend-style_learned_embeds.bin"
}

# Create styles directory in the Space and upload files
styles_dir = Path("styles")
if styles_dir.exists():
    for local_path, filename in STYLE_FILES.items():
        local_file = Path(local_path)
        if local_file.exists():
            api.upload_file(
                path_or_fileobj=str(local_file),
                path_in_repo=f"styles/{filename}",
                repo_id=REPO_ID,
                repo_type="space"
            )
            print(f"Uploaded: {local_path}")
        else:
            print(f"Warning: {local_path} not found, skipping...")
else:
    print("Warning: 'styles/' directory not found!")
    print("Please create a 'styles/' directory with your .bin embedding files.")
    print("Expected files:")
    for path in STYLE_FILES.keys():
        print(f"  - {path}")

In [ ]:
# Print the Space URL
print("=" * 60)
print("Deployment Complete!")
print("=" * 60)
print(f"\nYour Space is available at:")
print(f"https://huggingface.co/spaces/{REPO_ID}")
print("\nNote: It may take a few minutes for the Space to build and start.")
print("Check the 'Logs' tab on the Space page for build status.")

## (Optional) Test Locally

Run this cell to test the Gradio app locally before deploying.

In [ ]:
# Test the Gradio app locally/in Colab before deploying
# Uncomment the appropriate line and run (requires style files in styles/ directory)

# For Google Colab - launches with public shareable URL
# demo.launch(share=True, debug=True)

# For local Jupyter - opens in browser
# demo.launch()